# Pipeline ELT Final (Otimizado com Normalização em Memória e Logs em Tempo Real)
Este notebook executa o processo de ELT carregando os dados na staging `stg_samp_bruto` usando um método de bulk-copy (COPY) extremamente veloz. Para evitar o gargalo de performance no banco de dados, os dados são limpos e normalizados em memória (Pandas) antes do upload, garantindo que as queries SQL de criação das tabelas de dimensões e da tabela fato rodem em segundos e utilizem corretamente os índices do Postgres. As transformações foram calibradas para ficarem 100% idênticas ao resultado do processo ETL.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
import time

usuario_db = 'usuario_db'
senha_db = 'senha_db'
host_db = 'host_db'
porta_db = 'porta_db'
nome_banco_db = 'nome_banco_db'

DATABASE_URL = f'postgresql://{usuario_db}:{senha_db}@{host_db}:{porta_db}/{nome_banco_db}?client_encoding=utf8'
engine = create_engine(DATABASE_URL)

In [7]:
import io
import csv

# Método ultra-rápido para bulk insert no PostgreSQL usando a instrução COPY
def psql_insert_copy(table, conn, keys, data_iter):
    dbapi_conn = conn.connection
    # Garante compatibilidade tanto com SQLAlchemy 1.4 quanto 2.0+
    if hasattr(dbapi_conn, 'dbapi_connection'):
        dbapi_conn = dbapi_conn.dbapi_connection
        
    with dbapi_conn.cursor() as cur:
        s_buf = io.StringIO()
        writer = csv.writer(s_buf)
        writer.writerows(data_iter)
        s_buf.seek(0)
        
        columns = ', '.join('"{}"'.format(k) for k in keys)
        table_name = table.name
        if table.schema:
            table_name = f"{table.schema}.{table_name}"
            
        sql = f'COPY "{table_name}" ({columns}) FROM STDIN WITH CSV'
        cur.copy_expert(sql=sql, file=s_buf)

start_time = time.time()
arquivos_path = Path("..") / "Datas"
arquivos_name_list = sorted([arquivo.name for arquivo in arquivos_path.glob("*.csv")])

print("Resetando ambiente de Staging...")
with engine.connect() as conexao:
    conexao.execute(text("DROP TABLE IF EXISTS stg_samp_bruto CASCADE;"))
    conexao.commit()

print("Carregando e tratando dados para Staging...")
for arquivo_name in arquivos_name_list:
    t0 = time.time()
    path = arquivos_path / arquivo_name
    tdf = pd.read_csv(path, sep=";", encoding="cp1252", dtype=str)
    
    # 1. Limpa colunas e remove espaços em branco das pontas
    tdf.columns = tdf.columns.str.strip()
    
    # 2. Padroniza textos (Trim, Upper, remove espaços duplos)
    for col in tdf.select_dtypes(include=['object']).columns:
        tdf[col] = tdf[col].str.strip().str.upper()
        tdf[col] = tdf[col].str.replace(r"\s+", " ", regex=True)
        
    # 3. Padroniza termos específicos
    padronizacao_termos = {
        "NAO SE APLICA": "NÃO SE APLICA",
        "NÃO SE APLICA": "NÃO SE APLICA",
        "REGULAR": "REGULAR",
        "SISTEMA DE COMPENSAÇÃO GD I": "SISTEMA DE COMPENSAÇÃO GD I"
    }
    colunas_para_ajuste = ["NomTipoMercado", "DscClasseConsumoMercado", "DscModalidadeTarifaria"]
    for col in colunas_para_ajuste:
        if col in tdf.columns:
            tdf[col] = tdf[col].replace(padronizacao_termos)
            
    # 4. Tratamento de CNPJs
    for col in ["NumCNPJAgenteDistribuidora", "NumCNPJAgenteAcessante"]:
        if col in tdf.columns:
            tdf[col] = tdf[col].str.replace(r"\D", "", regex=True).replace({"": None})
            
    # 5. Tratamento de Acessante nulos
    tdf["IdeAgenteAcessante"] = tdf["IdeAgenteAcessante"].fillna("Nao Informado")
    tdf["NomAgenteAcessante"] = tdf["NomAgenteAcessante"].fillna("Nao Informado")
    tdf["NumCNPJAgenteAcessante"] = tdf["NumCNPJAgenteAcessante"].fillna("00000000000000")
    
    tdf["arquivo_origem"] = arquivo_name.upper()
    
    # Escreve na staging usando o método COPY otimizado
    tdf.to_sql('stg_samp_bruto', con=engine, if_exists='append', index=False, method=psql_insert_copy)
    print(f"-> Arquivo {arquivo_name} carregado na Staging em {time.time() - t0:.2f}s!")

# Adiciona coluna de auto-incremento para preservar a ordem exata de inserção do ETL
print("Adicionando coluna de identificação sequencial...")
with engine.connect() as conexao:
    conexao.execute(text("ALTER TABLE stg_samp_bruto ADD COLUMN staging_id SERIAL;"))
    conexao.commit()

print(f"Carga bruta em Staging concluída em {time.time() - start_time:.2f}s!")

Resetando ambiente de Staging...
Carregando e tratando dados para Staging...
-> Arquivo samp-2024.csv carregado na Staging em 53.48s!
-> Arquivo samp-2025.csv carregado na Staging em 52.44s!
-> Arquivo samp-2026.csv carregado na Staging em 14.56s!
Adicionando coluna de identificação sequencial...
Carga bruta em Staging concluída em 205.60s!


In [8]:
query_dim_tempo = """
DROP TABLE IF EXISTS dim_tempo CASCADE;
CREATE TABLE dim_tempo (
    tempo_id INT PRIMARY KEY,
    data_competencia DATE,
    ano INT,
    mes INT,
    nome_mes VARCHAR(20),
    trimestre INT,
    semestre INT
);

INSERT INTO dim_tempo (tempo_id, data_competencia, ano, mes, nome_mes, trimestre, semestre)
SELECT 
    ROW_NUMBER() OVER (ORDER BY data_bruta) as tempo_id,
    data_bruta as data_competencia,
    EXTRACT(YEAR FROM data_bruta)::INT as ano,
    EXTRACT(MONTH FROM data_bruta)::INT as mes,
    CASE EXTRACT(MONTH FROM data_bruta)::INT
        WHEN 1 THEN 'January'
        WHEN 2 THEN 'February'
        WHEN 3 THEN 'March'
        WHEN 4 THEN 'April'
        WHEN 5 THEN 'May'
        WHEN 6 THEN 'June'
        WHEN 7 THEN 'July'
        WHEN 8 THEN 'August'
        WHEN 9 THEN 'September'
        WHEN 10 THEN 'October'
        WHEN 11 THEN 'November'
        WHEN 12 THEN 'December'
    END as nome_mes,
    EXTRACT(QUARTER FROM data_bruta)::INT as trimestre,
    CASE WHEN EXTRACT(MONTH FROM data_bruta) <= 6 THEN 1 ELSE 2 END as semestre
FROM (
    SELECT DISTINCT TO_DATE("DatCompetencia", 'YYYY-MM-DD') as data_bruta
    FROM stg_samp_bruto
    WHERE "DatCompetencia" IS NOT NULL
) sub
ORDER BY data_bruta;
"""

query_dim_distribuidora = """
DROP TABLE IF EXISTS dim_distribuidora CASCADE;
CREATE TABLE dim_distribuidora (
    distribuidora_id INT PRIMARY KEY,
    "NumCNPJAgenteDistribuidora" VARCHAR(50),
    "SigAgenteDistribuidora" VARCHAR(50),
    "NomAgenteDistribuidora" VARCHAR(255)
);

INSERT INTO dim_distribuidora (distribuidora_id, "NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora")
SELECT 
    ROW_NUMBER() OVER (ORDER BY "SigAgenteDistribuidora", "NomAgenteDistribuidora") as distribuidora_id,
    "NumCNPJAgenteDistribuidora",
    "SigAgenteDistribuidora",
    "NomAgenteDistribuidora"
FROM (
    SELECT DISTINCT
        "NumCNPJAgenteDistribuidora",
        "SigAgenteDistribuidora",
        "NomAgenteDistribuidora"
    FROM stg_samp_bruto
    WHERE "NumCNPJAgenteDistribuidora" IS NOT NULL
) sub
ORDER BY "SigAgenteDistribuidora", "NomAgenteDistribuidora";
"""

query_dim_acessante = """
DROP TABLE IF EXISTS dim_acessante CASCADE;
CREATE TABLE dim_acessante (
    acessante_id INT PRIMARY KEY,
    "NumCNPJAgenteAcessante" VARCHAR(50),
    "IdeAgenteAcessante" VARCHAR(100),
    "NomAgenteAcessante" VARCHAR(255)
);

INSERT INTO dim_acessante (acessante_id, "NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante")
SELECT 
    ROW_NUMBER() OVER (ORDER BY "NomAgenteAcessante", "NumCNPJAgenteAcessante") as acessante_id,
    "NumCNPJAgenteAcessante",
    "IdeAgenteAcessante",
    "NomAgenteAcessante"
FROM (
    SELECT DISTINCT
        "NumCNPJAgenteAcessante",
        "IdeAgenteAcessante",
        "NomAgenteAcessante"
    FROM stg_samp_bruto
) sub
ORDER BY "NomAgenteAcessante", "NumCNPJAgenteAcessante";
"""

query_dim_mercado = """
DROP TABLE IF EXISTS dim_mercado CASCADE;
CREATE TABLE dim_mercado (
    mercado_id INT PRIMARY KEY,
    "NomTipoMercado" VARCHAR(150),
    "DscModalidadeTarifaria" VARCHAR(150),
    "DscSubGrupoTarifario" VARCHAR(150),
    "DscClasseConsumoMercado" VARCHAR(150),
    "DscSubClasseConsumidor" VARCHAR(150),
    "DscDetalheConsumidor" VARCHAR(150),
    "DscPostoTarifario" VARCHAR(150),
    "DscOpcaoEnergia" VARCHAR(150),
    "DscDetalheMercado" VARCHAR(150)
);

INSERT INTO dim_mercado (
    mercado_id, 
    "NomTipoMercado", 
    "DscModalidadeTarifaria", 
    "DscSubGrupoTarifario", 
    "DscClasseConsumoMercado", 
    "DscSubClasseConsumidor", 
    "DscDetalheConsumidor", 
    "DscPostoTarifario", 
    "DscOpcaoEnergia", 
    "DscDetalheMercado"
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY "NomTipoMercado", "DscModalidadeTarifaria", "DscPostoTarifario", "DscDetalheMercado", "DscSubGrupoTarifario", "DscClasseConsumoMercado", "DscSubClasseConsumidor", "DscDetalheConsumidor", "DscOpcaoEnergia") as mercado_id,
    "NomTipoMercado",
    "DscModalidadeTarifaria",
    "DscSubGrupoTarifario",
    "DscClasseConsumoMercado",
    "DscSubClasseConsumidor",
    "DscDetalheConsumidor",
    "DscPostoTarifario",
    "DscOpcaoEnergia",
    "DscDetalheMercado"
FROM (
    SELECT DISTINCT
        "NomTipoMercado",
        "DscModalidadeTarifaria",
        "DscSubGrupoTarifario",
        "DscClasseConsumoMercado",
        "DscSubClasseConsumidor",
        "DscDetalheConsumidor",
        "DscPostoTarifario",
        "DscOpcaoEnergia",
        "DscDetalheMercado"
    FROM stg_samp_bruto
) sub
ORDER BY "NomTipoMercado", "DscModalidadeTarifaria", "DscPostoTarifario", "DscDetalheMercado", "DscSubGrupoTarifario", "DscClasseConsumoMercado", "DscSubClasseConsumidor", "DscDetalheConsumidor", "DscOpcaoEnergia";
"""

start_time = time.time()
with engine.connect() as conexao:
    # 1. Dimensão Tempo
    print("Processando Dimensão Tempo...")
    t0 = time.time()
    conexao.execute(text(query_dim_tempo))
    conexao.commit()
    print(f"-> Dimensão Tempo concluída em {time.time() - t0:.2f}s!")
    
    # 2. Dimensão Distribuidora
    print("Processando Dimensão Distribuidora...")
    t0 = time.time()
    conexao.execute(text(query_dim_distribuidora))
    conexao.commit()
    print(f"-> Dimensão Distribuidora concluída em {time.time() - t0:.2f}s!")
    
    # 3. Dimensão Acessante
    print("Processando Dimensão Acessante...")
    t0 = time.time()
    conexao.execute(text(query_dim_acessante))
    conexao.commit()
    print(f"-> Dimensão Acessante concluída em {time.time() - t0:.2f}s!")
    
    # 4. Dimensão Mercado
    print("Processando Dimensão Mercado...")
    t0 = time.time()
    conexao.execute(text(query_dim_mercado))
    conexao.commit()
    print(f"-> Dimensão Mercado concluída em {time.time() - t0:.2f}s!")

print(f"\nTodas as tabelas de dimensões foram carregadas em {time.time() - start_time:.2f}s!")

Processando Dimensão Tempo...
-> Dimensão Tempo concluída em 40.42s!
Processando Dimensão Distribuidora...
-> Dimensão Distribuidora concluída em 4.72s!
Processando Dimensão Acessante...
-> Dimensão Acessante concluída em 4.89s!
Processando Dimensão Mercado...
-> Dimensão Mercado concluída em 123.72s!

Todas as tabelas de dimensões foram carregadas em 173.76s!


In [9]:
start_time = time.time()
query_criar_indices = """
CREATE INDEX IF NOT EXISTS idx_stg_competencia ON stg_samp_bruto ("DatCompetencia");
CREATE INDEX IF NOT EXISTS idx_stg_distribuidora ON stg_samp_bruto ("NumCNPJAgenteDistribuidora", "SigAgenteDistribuidora", "NomAgenteDistribuidora");
CREATE INDEX IF NOT EXISTS idx_stg_acessante ON stg_samp_bruto ("NumCNPJAgenteAcessante", "IdeAgenteAcessante", "NomAgenteAcessante");
CREATE INDEX IF NOT EXISTS idx_stg_mercado ON stg_samp_bruto ("NomTipoMercado", "DscModalidadeTarifaria", "DscSubGrupoTarifario", "DscClasseConsumoMercado", "DscSubClasseConsumidor", "DscDetalheConsumidor", "DscPostoTarifario", "DscOpcaoEnergia", "DscDetalheMercado");
"""

print("Criando índices tradicionais no banco de dados...")
with engine.connect() as conexao:
    conexao.execute(text(query_criar_indices))
    conexao.commit()
print(f"Índices de performance criados com sucesso em {time.time() - start_time:.2f}s!")

Criando índices tradicionais no banco de dados...
Índices de performance criados com sucesso em 130.83s!


In [ ]:
start_time = time.time()
query_fato = """
-- 5. TABELA FATO ENERGIA (Agora com joins simples e ultra-rápidos sem computação on-the-fly)
DROP TABLE IF EXISTS fato_energia CASCADE;
CREATE TABLE fato_energia (
    fato_id INT PRIMARY KEY,
    tempo_id INT,
    distribuidora_id INT,
    acessante_id INT,
    mercado_id INT,
    "DatGeracaoConjuntoDados" DATE,
    "VlrMercado" NUMERIC(18,2),
    arquivo_origem VARCHAR(100)
);

INSERT INTO fato_energia (fato_id, tempo_id, distribuidora_id, acessante_id, mercado_id, "DatGeracaoConjuntoDados", "VlrMercado", arquivo_origem)
SELECT 
    ROW_NUMBER() OVER (ORDER BY s.staging_id) as fato_id,
    t.tempo_id,
    d.distribuidora_id,
    a.acessante_id,
    m.mercado_id,
    TO_DATE(s."DatGeracaoConjuntoDados", 'YYYY-MM-DD') as "DatGeracaoConjuntoDados",
    COALESCE(REPLACE(REPLACE(s."VlrMercado", '.', ''), ',', '.')::NUMERIC, 0) as "VlrMercado",
    s.arquivo_origem
FROM stg_samp_bruto s
LEFT JOIN dim_tempo t ON TO_DATE(s."DatCompetencia", 'YYYY-MM-DD') = t.data_competencia
LEFT JOIN dim_distribuidora d ON s."NumCNPJAgenteDistribuidora" = d."NumCNPJAgenteDistribuidora"
    AND s."SigAgenteDistribuidora" = d."SigAgenteDistribuidora"
    AND s."NomAgenteDistribuidora" = d."NomAgenteDistribuidora"
LEFT JOIN dim_acessante a ON s."NumCNPJAgenteAcessante" = a."NumCNPJAgenteAcessante"
    AND s."IdeAgenteAcessante" = a."IdeAgenteAcessante"
    AND s."NomAgenteAcessante" = a."NomAgenteAcessante"
LEFT JOIN dim_mercado m ON s."NomTipoMercado" = m."NomTipoMercado"
    AND s."DscModalidadeTarifaria" = m."DscModalidadeTarifaria"
    AND s."DscSubGrupoTarifario" = m."DscSubGrupoTarifario"
    AND s."DscClasseConsumoMercado" = m."DscClasseConsumoMercado"
    AND s."DscSubClasseConsumidor" = m."DscSubClasseConsumidor"
    AND s."DscDetalheConsumidor" = m."DscDetalheConsumidor"
    AND s."DscPostoTarifario" = m."DscPostoTarifario"
    AND s."DscOpcaoEnergia" = m."DscOpcaoEnergia"
    AND s."DscDetalheMercado" = m."DscDetalheMercado"
WHERE s."DatCompetencia" IS NOT NULL 
  AND s."VlrMercado" IS NOT NULL
  AND s."VlrMercado" IS DISTINCT FROM ''
  AND t.tempo_id IS NOT NULL
  AND d.distribuidora_id IS NOT NULL
  AND a.acessante_id IS NOT NULL
  AND m.mercado_id IS NOT NULL;
"""

print("Executando a query de relacionamento e inserção de 3 milhões de registros na Fato (isso deve levar entre 5 e 15 segundos)...\n")
with engine.connect() as conexao:
    conexao.execute(text(query_fato))
    conexao.commit()
print(f"Tabela fato criada e populada com sucesso em {time.time() - start_time:.2f}s!")

Executando a query de relacionamento e inserção de 3 milhões de registros na Fato (isso deve levar entre 5 e 15 segundos)...



In [ ]:
# Célula de validação do resultado do ELT com o ETL
with engine.connect() as conexao:
    res_tempo = conexao.execute(text("SELECT COUNT(*) FROM dim_tempo;")).scalar()
    res_dist = conexao.execute(text("SELECT COUNT(*) FROM dim_distribuidora;")).scalar()
    res_aces = conexao.execute(text("SELECT COUNT(*) FROM dim_acessante;")).scalar()
    res_merc = conexao.execute(text("SELECT COUNT(*) FROM dim_mercado;")).scalar()
    res_fato = conexao.execute(text("SELECT COUNT(*) FROM fato_energia;")).scalar()

print(f"Valores resultantes do ELT final:")
print(f"Dim tempo linhas: {res_tempo}")
print(f"Dim distribuidora linhas: {res_dist}")
print(f"Dim acessante linhas: {res_aces}")
print(f"Dim mercado linhas: {res_merc}")
print(f"Fato energia linhas: {res_fato}")

Valores resultantes do ELT final:
Dim tempo linhas: 28
Dim distribuidora linhas: 103
Dim acessante linhas: 1643
Dim mercado linhas: 12902
Fato energia linhas: 3029329
